<a href="https://colab.research.google.com/github/Siddarth-Velan/ML-Work/blob/ACM-Work/Dec_Trees_and_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter
from sklearn import datasets

In [2]:
class Node():
  def __init__(self, feature = None, thresh = None, right = None, left = None,*, value = None):
    self.feature = feature
    self.thresh = thresh
    self.right = right
    self.left = left
    self.value = value
  def check(self):
    return self.value is not None

class Dec_tree():
  def __init__(self, min_samples_split = 2, max_d = 100, n_feat = None):
    self.min_samples_split = min_samples_split
    self.max_d = max_d
    self.n_feat = n_feat
    self.root = None
  def fit(self, X, y):
    self.X = X
    self.y = y
    self.n_feat = X.shape[1]
    self.root = self._grow_tree(X, y)


  def _grow_tree(self, X, y, depth = 0):
    n_samples, n_feats = X.shape
    n_labels = len(np.unique(y))
    if (depth >= self.max_d or n_labels ==1 or n_samples < self.min_samples_split):
      leaf_val = self._most_common_label(y)
      return Node(value = leaf_val)
    feat_idx = np.random.choice(n_feats, self.n_feat, replace = False)
    best_feat, best_thresh = self._best_split(X, y, feat_idx)

    left_idx, right_idx = self._split(X[:, best_feat], best_thresh)
    left = self._grow_tree(X[left_idx, :], y[left_idx], depth+1)
    right = self._grow_tree(X[right_idx, :], y[right_idx], depth+1)

    return Node(best_feat, best_thresh, right, left)
  def _best_split(self, X, y, feat_idx):
    best_gain = -1
    split_idx, split_thresh = None, None
    for idx in feat_idx:
      X_c = X[:, idx]
      thresholds = np.unique(X_c)
      for t in thresholds:
        gain = self._IG(y, X_c, t)
        if gain>best_gain:
          best_gain = gain
          split_idx = idx
          split_thresh = t
    return split_idx, split_thresh
  def _IG(self, y, X_c, thresholds):
    parent_ent = self._Entropy(y)
    left_idx, right_idx = self._split(X_c, thresholds)
    if len(left_idx) == 0  or len(right_idx) == 0 :
      return 0
    n_l, n_r = len(left_idx), len(right_idx)
    e_l, e_r = self._Entropy(y[left_idx]), self._Entropy(y[right_idx])
    child_ent = (n_l / len(y))*e_l + (n_r/len(y))*e_r
    return parent_ent - child_ent
  def _split(self, X_c, split_thresh):
    left_idxs = np.argwhere(X_c <= split_thresh).flatten()
    right_idxs = np.argwhere(X_c > split_thresh).flatten()
    return left_idxs, right_idxs
  def _Entropy(self, y):
    h = np.bincount(y) / len(y)
    return -np.sum([i * np.log2(i) for i in h if i>0])
  def _most_common_label(self, y):
    count = Counter(y)
    value = count.most_common(1)[0][0]
    return value
  def _traverse_tree(self, X, node):
    if node.check():
      return node.value
    if X[node.feature]<= node.thresh:
      return self._traverse_tree(X, node.left)
    return self._traverse_tree(X, node.right)
  def predict(self, X):
    return np.array([self._traverse_tree(x, self.root) for x in X])

In [3]:
data = datasets.load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 67)


In [4]:
dec = Dec_tree(max_d = 15)
dec.fit(X_train, y_train)
preds = dec.predict(X_test)

In [5]:
def accuracy(y_test, y_pred):
  return np.sum(y_test == y_pred) / len(y_test)

In [6]:
accuracy(y_test, preds)

np.float64(0.9385964912280702)

In [9]:
class Random_forest():
  def __init__(self, n_trees = 10, min_samples_split = 2, max_d = 100, n_feat = None):
    self.n_trees = n_trees
    self.min_samples_split = min_samples_split
    self.max_d = max_d
    self.n_feat = n_feat
    self.trees = []

  def fit(self, X, y):
    for i in range(self.n_trees):
      tree = Dec_tree(min_samples_split = self.min_samples_split, max_d = self.max_d, n_feat = self.n_feat )
      X_s, y_s = self._random_samples(X,y)
      tree.fit(X_s, y_s)
      self.trees.append(tree)
  def _random_samples(self, X, y):
    n = X.shape[0]
    idxs = np.random.choice(n, n, replace = True)
    return X[idxs], y[idxs]
  def _most_common_label(self, y):
    count = Counter(y)
    value = count.most_common(1)[0][0]
    return value
  def predict(self, X):
    pred = np.array([tree.predict(X) for tree in self.trees])
    t_pred = np.swapaxes(pred, 0, 1)
    return np.array([self._most_common_label(preds) for preds in t_pred])

In [10]:
forest = Random_forest()
forest.fit(X_train, y_train)
preds2 = forest.predict(X_test)
accuracy(y_test, preds2)

np.float64(0.9649122807017544)